In [33]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 
from uuid import uuid4
import json

### Build tool registry JSON from function definitions.
The LLM can search the registry, fetch and execute scripts, and push new scripts to the registry


In [ ]:
"""Persistent registration and execution of generated Python function tools."""

from __future__ import annotations

import ast
from contextlib import contextmanager
import fcntl
import hashlib
import json
import keyword
import os
from pathlib import Path
import re
import subprocess
import sys
import tempfile
from typing import Union
from uuid import uuid4

from pydantic import ConfigDict, ValidationError, create_model


PROJECT_ROOT = Path(__file__).resolve().parents[1]


class RegistryError(ValueError):
    """An actionable registration, validation, or execution failure."""


def _annotation(node):
    """Resolve a small JSON-compatible type vocabulary without eval()."""
    primitives = {"str": str, "int": int, "float": float, "bool": bool}
    if isinstance(node, ast.Constant):
        if node.value is None:
            return type(None)
        if isinstance(node.value, str):
            return _annotation(ast.parse(node.value, mode="eval").body)
    if isinstance(node, ast.Name) and node.id in primitives:
        return primitives[node.id]
    if isinstance(node, ast.BinOp) and isinstance(node.op, ast.BitOr):
        return Union[_annotation(node.left), _annotation(node.right)]
    if isinstance(node, ast.Subscript) and isinstance(node.value, ast.Name):
        if node.value.id == "list":
            return list[_annotation(node.slice)]
        if node.value.id == "dict" and isinstance(node.slice, ast.Tuple):
            keys, values = node.slice.elts
            if _annotation(keys) is str:
                return dict[str, _annotation(values)]
    raise RegistryError(
        "Use JSON-compatible annotations: str, int, float, bool, list[T], "
        "dict[str, T], or unions such as str | None."
    )


def extract_source_metadata(source: str, entry_point: str):
    """Return function metadata and a strict argument validator; never import source."""
    try:
        tree = ast.parse(source)
        compile(tree, "<generated-script>", "exec")
    except (SyntaxError, ValueError, TypeError) as exc:
        raise RegistryError(f"Invalid Python source: {exc}") from exc
    matches = [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and node.name == entry_point
    ]
    if len(matches) != 1:
        raise RegistryError("Source must define exactly one top-level entry-point function.")
    function = matches[0]
    if isinstance(function, ast.AsyncFunctionDef) or function.decorator_list:
        raise RegistryError("Use a synchronous, undecorated entry-point function.")
    args = function.args
    if args.posonlyargs or args.vararg or args.kwarg:
        raise RegistryError("Entry points must use named parameters, without *args or **kwargs.")
    description = ast.get_docstring(function)
    if not description:
        raise RegistryError("The entry-point function needs a descriptive docstring.")
    fields = {}
    defaults = [None] * (len(args.args) - len(args.defaults)) + list(args.defaults)
    parameters = list(zip(args.args, defaults)) + list(zip(args.kwonlyargs, args.kw_defaults))
    try:
        for parameter, default in parameters:
            if parameter.annotation is None:
                raise RegistryError(f"Parameter {parameter.arg!r} needs a type annotation.")
            value = ... if default is None else ast.literal_eval(default)
            if default is not None:
                json.dumps(value, allow_nan=False)
            fields[parameter.arg] = (_annotation(parameter.annotation), value)
        argument_model = create_model(
            "ScriptArguments",
            __config__=ConfigDict(extra="forbid", strict=True, validate_default=True),
            **fields,
        )
        if function.returns is None:
            raise RegistryError("The entry-point function needs a return annotation.")
        _annotation(function.returns)
    except RegistryError:
        raise
    except (ValueError, TypeError, SyntaxError) as exc:
        raise RegistryError(f"Unsupported parameter annotation or default: {exc}") from exc
    return {
        "name": function.name,
        "description": description,
        "parameters": argument_model.model_json_schema(),
        "return_type": ast.unparse(function.returns),
    }, argument_model


class ScriptRegistry:
    """A project-local JSON registry. Registration does not enable execution."""

    def __init__(self, project_root: Path | str = PROJECT_ROOT):
        self.root = Path(project_root).resolve()
        self.registry_path = self.root / "tools" / "registry.json"
        self.scripts_dir = self.root / "agent_generated_scripts"

    @contextmanager
    def _lock(self):
        self.registry_path.parent.mkdir(parents=True, exist_ok=True)
        with self.registry_path.with_suffix(".lock").open("a") as lock:
            fcntl.flock(lock, fcntl.LOCK_EX)
            try:
                yield
            finally:
                fcntl.flock(lock, fcntl.LOCK_UN)

    def _read(self):
        if not self.registry_path.exists():
            return {"tools": []}
        try:
            data = json.loads(self.registry_path.read_text())
        except (OSError, json.JSONDecodeError) as exc:
            raise RegistryError(f"Cannot read registry: {exc}") from exc
        if not isinstance(data, dict) or not isinstance(data.get("tools"), list):
            raise RegistryError("Registry must contain a 'tools' list.")
        if not all(isinstance(entry, dict) for entry in data["tools"]):
            raise RegistryError("Every registry entry must be an object.")
        return data

    def _write(self, data):
        """Replace the JSON file only after a complete, flushed write."""
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", dir=self.registry_path.parent, suffix=".tmp", delete=False
            ) as stream:
                temporary = Path(stream.name)
                json.dump(data, stream, indent=2, allow_nan=False)
                stream.write("\n")
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temporary, self.registry_path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    def register(self, script_name: str, source: str, entry_point: str) -> dict:
        if not script_name.isidentifier() or keyword.iskeyword(script_name):
            raise RegistryError("script_name must be a Python identifier, without a path or extension.")
        metadata, _ = extract_source_metadata(source, entry_point)
        source_bytes = source.encode("utf-8")
        with self._lock():
            data = self._read()
            if any(entry.get("name") == metadata["name"] for entry in data["tools"]):
                raise RegistryError(f"A tool named {metadata['name']!r} is already registered.")
            self.scripts_dir.mkdir(parents=True, exist_ok=True)
            path = self.scripts_dir / f"{script_name}.py"
            metadata.update({
                "id": str(uuid4()),
                "path": str(path.relative_to(self.root)),
                "entry_point": entry_point,
                "status": "pending_validation",
                "version": 1,
                "sha256": hashlib.sha256(source_bytes).hexdigest(),
            })
            try:
                with path.open("xb") as stream:
                    try:
                        stream.write(source_bytes)
                        stream.flush()
                        os.fsync(stream.fileno())
                    except BaseException:
                        path.unlink(missing_ok=True)
                        raise
            except FileExistsError as exc:
                raise RegistryError(f"Script already exists: {path.name}") from exc
            try:
                data["tools"].append(metadata)
                self._write(data)
            except BaseException:
                path.unlink(missing_ok=True)
                raise
        return metadata

    def search(self, query: str, limit: int = 5) -> list[dict]:
        if not 1 <= limit <= 20:
            raise RegistryError("limit must be between 1 and 20.")
        words = set(re.findall(r"\w+", query.lower()))
        entries = self._read()["tools"]
        def score(entry):
            text = f"{entry.get('name', '')} {entry.get('description', '')}".lower()
            return sum(word in text for word in words)
        matches = [entry for entry in entries if not words or score(entry)]
        return sorted(matches, key=score, reverse=True)[:limit]

    def _find(self, data, script_id):
        for entry in data["tools"]:
            if entry.get("id") == script_id:
                return entry
        raise RegistryError(f"Unknown script ID: {script_id}")

    def _verified_path(self, entry):
        if not entry.get("path") or not entry.get("sha256"):
            raise RegistryError("Legacy metadata must be registered with source before execution.")
        path = (self.root / entry["path"]).resolve()
        if not path.is_relative_to(self.scripts_dir.resolve()) or path.suffix != ".py":
            raise RegistryError("Registered script path is outside the scripts directory.")
        if not path.is_file():
            raise RegistryError("Registered script file is missing.")
        if hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
            raise RegistryError("Script changed after registration; review and register it again.")
        return path

    def enable_script(self, script_id: str) -> dict:
        """Enable a reviewed script. Deliberately not exposed as an LLM tool."""
        with self._lock():
            data = self._read()
            entry = self._find(data, script_id)
            self._verified_path(entry)
            entry["status"] = "enabled"
            self._write(data)
            return entry

    def execute(self, script_id: str, arguments: dict, timeout: float = 10) -> dict:
        entry = self._find(self._read(), script_id)
        if entry.get("status") != "enabled":
            raise RegistryError("Script is not enabled. Review it and call registry.enable_script(id).")
        path = self._verified_path(entry)
        _, validator = extract_source_metadata(path.read_text(), entry["entry_point"])
        try:
            validated = validator.model_validate(arguments, strict=True)
            payload = json.dumps({
                "path": str(path), "entry_point": entry["entry_point"],
                "arguments": validated.model_dump(mode="json"),
            }, allow_nan=False)
        except (ValidationError, TypeError, ValueError) as exc:
            raise RegistryError(f"Invalid arguments: {exc}") from exc
        try:
            process = subprocess.run(
                [sys.executable, "-I", str(Path(__file__).with_name("tool_runner.py"))],
                input=payload, text=True, capture_output=True,
                cwd=self.root, timeout=timeout,
            )
        except subprocess.TimeoutExpired as exc:
            raise RegistryError(f"Script exceeded the {timeout}-second timeout.") from exc
        if process.returncode:
            raise RegistryError(f"Script failed: {process.stderr.strip()[-4000:]}")
        try:
            return json.loads(process.stdout)
        except json.JSONDecodeError as exc:
            raise RegistryError("Script runner returned invalid JSON.") from exc


def build_registry_tools(registry: ScriptRegistry | None = None):
    """Build LangChain tools backed by one persistent registry instance."""
    from langchain_core.tools import tool

    registry = registry or ScriptRegistry()

    @tool
    def write_script_tool(script_name: str, source: str, entry_point: str) -> dict:
        """Save a Python function tool with typed parameters, return type and docstring.

        script_name is a simple name without .py. entry_point names a top-level
        synchronous function. Registration returns an ID and pending_validation
        status; a reviewed script must be enabled before it can execute.
        """
        try:
            return registry.register(script_name, source, entry_point)
        except (RegistryError, OSError) as exc:
            return {"error": str(exc)}

    @tool
    def search_scripts(query: str, limit: int = 5) -> list[dict]:
        """Search saved script names and descriptions; return IDs, schemas and status."""
        try:
            return registry.search(query, limit)
        except RegistryError as exc:
            return [{"error": str(exc)}]

    @tool
    def execute_script(script_id: str, arguments: dict) -> dict:
        """Execute an enabled registered function by ID with JSON arguments.

        Returns its JSON result and captured stdout. Search the registry first
        to discover IDs and argument schemas. Execution has a ten-second timeout.
        """
        try:
            return registry.execute(script_id, arguments)
        except (RegistryError, OSError) as exc:
            return {"error": str(exc)}

    return [write_script_tool, search_scripts, execute_script]


In [ ]:
from src.tool_registry import ScriptRegistry, build_registry_tools

# The inspect example above demonstrates metadata extraction only.
# Persist new tools through write_script_tool, which parses source without running it.
registry = ScriptRegistry()
registry.register("print","print('hello')","my_print")


[{'id': 'd237288f-0134-4d3d-b061-c6f697ffdfff',
  'name': 'add_numbers',
  'description': 'a function to add two numbers\nargs:\n    a: first number\n    b: second number\nreturns: the sum a+b',
  'parameters': [{'name': 'a',
    'type': 'float',
    'kind': 'POSITIONAL_OR_KEYWORD',
    'required': True,
    'default': None},
   {'name': 'b',
    'type': 'float',
    'kind': 'POSITIONAL_OR_KEYWORD',
    'required': True,
    'default': None}],
  'return_type': 'float'}]

In [ ]:
from src.tool_registry import ScriptRegistry, build_registry_tools

registry = ScriptRegistry()
tools = build_registry_tools(registry)

# After reviewing a generated script, enable it explicitly:
# registry.enable_script("the-id-returned-by-write_script_tool")


In [ ]:
chat = ChatOllama(
    model="qwen3.5:4b",
    temperature=0,
    max_tokens=2048,
)

SystemPrompt = """You are a voice assistant with a persistent Python tool registry.
Search with search_scripts before creating a tool. Use execute_script with an
existing enabled tool's ID and schema-compatible arguments when it fits.
If no suitable tool exists, use write_script_tool to save a new script. Give
its entry-point function typed named parameters, a return annotation, a clear
docstring, and a JSON-serializable result. script_name has no path or extension.
New scripts are pending_validation: explain that review and enabling are needed
before execution. Never claim a pending script has run. Tool failures are
returned as error objects; read them and correct the request when appropriate.
"""
tools = build_registry_tools(registry)

messages = [HumanMessage(content="Print the word Hell after 3 seconds.")]
messages_write_python_script = [HumanMessage(
    content="Create and register a reusable function that takes two numbers and returns their sum."
)]


def llm_chat(messages: list, tools: list | None = None, max_rounds: int = 8) -> list:
    available = {registered_tool.name: registered_tool for registered_tool in (tools or [])}
    model_with_tools = chat.bind_tools(list(available.values())) if available else chat
    for _ in range(max_rounds):
        result = model_with_tools.invoke([SystemMessage(content=SystemPrompt)] + messages)
        messages.append(result)
        if result.content:
            print(f"[AGENT]: {result.content}")
        if not result.tool_calls:
            return messages
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call['name']}")
            try:
                registered_tool = available[call["name"]]
                tool_result = registered_tool.invoke(call["args"])
            except Exception as exc:
                tool_result = {"error": str(exc)}
            print(f"[OBS]: {tool_result}")
            messages.append(ToolMessage(
                content=json.dumps(tool_result), tool_call_id=call["id"]
            ))
    raise RuntimeError("Agent reached its tool-call round limit.")


In [4]:
result = llm_chat(messages_write_python_script, tools)

[TOOL CALL]: {'name': 'write_python_script', 'args': {'path': 'agent_generated_scripts/sum_calculator.py', 'content': '#!/usr/bin/env python3\n"""Sum calculator that takes two arguments and returns their sum."""\n\ndef calculate_sum(a, b):\n    """Calculate the sum of two numbers."""\n    return a + b\n\nif __name__ == "__main__":\n    import sys\n    \n    if len(sys.argv) != 3:\n        print("Usage: python sum_calculator.py <num1> <num2>")\n        sys.exit(1)\n    \n    try:\n        num1 = float(sys.argv[1])\n        num2 = float(sys.argv[2])\n        result = calculate_sum(num1, num2)\n        print(f"Sum of {num1} and {num2} is: {result}")\n    except ValueError:\n        print("Error: Please provide two valid numbers as arguments.")\n        sys.exit(1)'}, 'id': 'bec4bdc7-9d7e-4a7b-8518-2e88572feb3b', 'type': 'tool_call'}
[OBS]: null


#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [5]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



/home/bashar/Study/Projects/AIVoiceAssistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: huggingface-hub>=0.34.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==2.2.0.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recording audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sr),
        samplerate=sr,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [ ]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 0 ns, sys: 14 μs, total: 14 μs
Wall time: 25 μs
[ASRTranscription(language='English', text='Hello, what is the capital of France?', time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output

In [ ]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = llm_chat(messages=messages, tools=tools)


Recroding audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: The "secret of the universe" is something that has fascinated humanity for millennia, but there isn't one single answer that everyone agrees on. Here are a few perspectives:

**From a scientific standpoint:**
- The universe appears to be expanding at an accelerating rate, driven by dark energy
- We're still searching for what constitutes dark matter and dark energy (together they make up about 95% of the universe)
- The fundamental constants of physics seem fine-tuned for life

**From a philosophical standpoint:**
- Some suggest consciousness itself is the secret - that we are the universe experiencing itself
- Others believe the true secret lies in understanding our place within the cosmos and finding meaning in existence

**From an existential perspective:**
- Perhaps the "secret" is simply that we exist at all, and that the act of asking questions is what makes us human

I don't have access to any hidden information or secrets - I'm an AI assistant designed to help with tas

In [ ]:
llm_result

[HumanMessage(content='What is the secret of the universe that no one else knows, and you can tell me it right now without any hesitation, please?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The "secret of the universe" is something that has fascinated humanity for millennia, but there isn\'t one single answer that everyone agrees on. Here are a few perspectives:\n\n**From a scientific standpoint:**\n- The universe appears to be expanding at an accelerating rate, driven by dark energy\n- We\'re still searching for what constitutes dark matter and dark energy (together they make up about 95% of the universe)\n- The fundamental constants of physics seem fine-tuned for life\n\n**From a philosophical standpoint:**\n- Some suggest consciousness itself is the secret - that we are the universe experiencing itself\n- Others believe the true secret lies in understanding our place within the cosmos and finding meaning in existence\n\n**From an existential perspective:**\n-

### The Pipeline looks fairly good, we will now deal with voice verification 

In [ ]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('recording.wav')
print(signal.shape)
signal = signal.mean(dim=0, keepdim=True)
signal = torchaudio.functional.resample(signal, fs, 16000) # necessary for ecapa-tdnn
embeddings = classifier.encode_batch(signal)


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([1, 120000])


In [ ]:
from speechbrain.inference.speaker import SpeakerRecognition
verification = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb")
score, prediction = verification.verify_files("recording.wav", "recording_dif.wav") # Different Speakers
print(f"{score= }, {prediction= }")
score, prediction = verification.verify_files("recording.wav", "recording_sim.wav") # Same Speaker
print(f"{score= }, {prediction= }")

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


score= tensor([0.2358], device='cuda:0'), prediction= tensor([False], device='cuda:0')
score= tensor([0.7795], device='cuda:0'), prediction= tensor([True], device='cuda:0')


In [ ]:

import torch
import torch.nn.functional as F 


def enroll_speaker(trials=2, trial_duration=5):
    pass_threshold = 0.45
    print(f"You will be asked to speak {trials} times, each time you can say whatever you want for {trial_duration} seconds, the model will save your voiceprint afterwards")
    embeddings = []
    for i in range(trials):
        input(f"Press Enter to Start trial {i+1}...")
        print(f"Trial number {i+1} : ")
        audio, sr = record_audio(16000, trial_duration)
        audio = torch.tensor(audio, dtype=torch.float32)
        audio = audio.T
        audio = audio.mean(dim=0, keepdim=True) # in case of multiple channels, ecapa accepts mono audio with sr = 16000
        audio = torchaudio.functional.resample(audio, sr, 16000)
        embeddings.append(classifier.encode_batch(audio, normalize=False))
        if i > 0:
            similarity = F.cosine_similarity(embeddings[-1][0,:,:], embeddings[-2][0,:,:])
            assert similarity.item() > pass_threshold, "Your recorded voice is very different from the previous, try again"
    res = torch.cat(embeddings, dim=0)
    res = F.normalize(res, p=2, dim=-1)
    print(f"Finished Enrollment Procedure")
    return F.normalize(res.mean(dim=0), dim=-1)



def authorized_speaker(audio, sr, ref_dim:torch.tensor) -> bool:
    """Checks the similarity with the enrolled speaker """
    if not isinstance(audio, torch.Tensor):
        audio = torch.tensor(audio, dtype=torch.float32)
        if audio.shape[0] != 1:
            audio = audio.T
    print(audio.shape)
    audio = audio.mean(dim=0, keepdim=True)
    audio = torchaudio.functional.resample(audio, sr, 16000)
    audio_dim = classifier.encode_batch(audio, normalize=False)[0]
    audio_dim = F.normalize(audio_dim, p=2, dim=-1)
    sim = F.cosine_similarity(audio_dim, ref_dim)
    print(f"{sim=}")
    return sim.item() > 0.45




In [ ]:
authorized_spk_dim = enroll_speaker(trials=3, trial_duration=5)

You will be asked to speak 3 times, each time you can say whatever you want for 5 seconds, the model will save your voiceprint afterwards
Trial number 1 : 
Recording audio for 5 seconds...
Trial number 2 : 
Recording audio for 5 seconds...


AssertionError: Your recorded voice is very different from the previous, try again

In [ ]:
audio = record_audio(16000, 5)

Recording audio for 5 seconds...


In [ ]:
authorized_speaker(audio[0], audio[1], authorized_spk_dim)

torch.Size([1, 120000])
sim=tensor([0.5490], device='cuda:0')


False

### The speaker verification is working fine for now (needs further testing to check the authorization).
### The next step is to add it to the pipeline : Audio -> Speaker authorization -> ASR -> LLM -> Response

In [ ]:
audio = record_audio(16000, 10)
if not authorized_speaker(audio[0], audio[1], authorized_spk_dim):
    raise Exception('Unauthorized speaker.')
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content=text)]
tools = build_registry_tools(registry)
response = llm_chat(messages=messages, tools=tools)

Go ahead and write a script Python that takes two arguments and returns their sum with a square root.

null


### 